# AuditoryNet on Google Colab (GPU)
Speech-based Parkinson's detection - Gammatone cochleagram CNN.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Put these two files in your Google Drive (top level *My Drive*):
   - `auditorynet_project.zip` (the project)
   - `dataset.zip` - contains the task folders directly: `E/PD`, `E/Healthy`, `I/PD`, ... (any nesting level is fine, it is auto-detected)
3. Run the cells **top to bottom**. Results are saved to Drive, so a disconnect will not lose finished tasks.

> Research prototype - not a medical device.

In [ ]:
# 1) Check the GPU
import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU! Use Runtime > Change runtime type > T4 GPU (CPU will be very slow).")

## 2) Settings - edit this cell only

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ======================= EDIT THESE =======================
PROJECT_ZIP = "/content/drive/MyDrive/auditorynet_project.zip"
DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"        # or None if you set DATASET_DIR
DATASET_DIR = None                                         # e.g. "/content/drive/MyDrive/my_dataset" (unzipped folder)
OUTPUT_DIR  = "/content/drive/MyDrive/auditorynet_outputs" # results + trained models are saved here

TASKS   = ["A", "E", "I", "O", "U", "petaka", "apto"]      # remove tasks you do not have
EPOCHS  = 25
SPEAKER_REGEX = None      # e.g. r"^(spk\d+)_"  -> speaker-disjoint folds (recommended if speakers repeat)
FORCE_RETRAIN = False     # True = retrain tasks that already have results in OUTPUT_DIR
RUN_QUICK_TEST = True     # 2-epoch smoke test on one task before the full run
# ==========================================================

## 3) Install the project and unpack the dataset

In [ ]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path

def run(cmd):
    """Run a command and stream its output live into the notebook. Returns the exit code."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="")
    return p.wait()

# project
# NOTE: the project folder must NOT be called "auditorynet" (it would shadow the Python package)
PROJ = "/content/auditorynet_project"
SRC = PROJ + "/src"
for d in (PROJ, "/content/auditorynet", "/content/_unzip"):
    shutil.rmtree(d, ignore_errors=True)
with zipfile.ZipFile(PROJECT_ZIP) as z:
    z.extractall("/content/_unzip")
roots = [p.parent for p in Path("/content/_unzip").rglob("pyproject.toml")]
assert roots, "pyproject.toml not found inside PROJECT_ZIP - check the file name/path"
shutil.move(str(roots[0]), PROJ)
os.chdir("/content")
sys.path.insert(0, SRC)                 # make `import auditorynet` work in this notebook
os.environ["PYTHONPATH"] = SRC          # ...and in the training subprocesses
for m in [k for k in sys.modules if k.startswith("auditorynet")]:
    del sys.modules[m]
assert Path(SRC, "auditorynet", "config.py").exists(), "src/auditorynet not found in the project zip"
assert run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "pyyaml", "pandas", "scikit-learn", "scipy", "matplotlib"]) == 0

# dataset
if DATASET_DIR:
    base = DATASET_DIR
else:
    base = "/content/data"
    shutil.rmtree(base, ignore_errors=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(base)

KNOWN = {"a", "e", "i", "o", "u", "petaka", "apto"}
def find_root(b):
    for dp, dn, _ in os.walk(b):
        if any(d.lower() in KNOWN for d in dn):
            return dp
    return None

DATA_DIR = find_root(base)
assert DATA_DIR, "Could not find task folders (E, I, O, U, A, petaka, apto) inside the dataset."
print("Dataset root:", DATA_DIR)

## 4) Check the dataset

In [ ]:
import pandas as pd
from auditorynet.gui_utils import scan_dataset

table = pd.DataFrame(scan_dataset(DATA_DIR, TASKS))
display(table)
usable = table.loc[table["Status"] == "OK", "Task"].tolist()
skipped = [t for t in TASKS if t not in usable]
if skipped:
    print("Not trainable (missing or too few files), will be skipped:", skipped)
TASKS = usable
assert TASKS, "No usable task. Each task needs PD/ and Healthy/ folders with >= 10 WAV files per class."

## 5) Quick smoke test (optional, ~1-2 min)

In [ ]:
if RUN_QUICK_TEST:
    rc = run([sys.executable, "-u", "-m", "auditorynet.train", "--data-dir", DATA_DIR,
              "--output-dir", "/content/quick_test", "--tasks", TASKS[0], "--epochs", "2",
              "--device", "cuda" if torch.cuda.is_available() else "cpu"])
    print("\nQuick test", "PASSED" if rc == 0 else f"FAILED (exit code {rc}) - copy the error above and send it for a fix")
    assert rc == 0
    shutil.rmtree("/content/quick_test", ignore_errors=True)

## 6) Full training (one task at a time; finished tasks are skipped after a disconnect)

In [ ]:
from auditorynet.gui_utils import write_config

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CFG = "/content/colab_config.yaml"
write_config({"data_dir": DATA_DIR, "output_dir": OUTPUT_DIR, "epochs": EPOCHS, "tasks": TASKS,
              "device": "cuda" if torch.cuda.is_available() else "cpu",
              "speaker_regex": SPEAKER_REGEX}, Path(CFG))

failed = []
for t in TASKS:
    if (Path(OUTPUT_DIR) / f"{t}_results.json").exists() and not FORCE_RETRAIN:
        print(f"[skip] {t}: results already in {OUTPUT_DIR}")
        continue
    print(f"\n================ TRAINING TASK {t} ================")
    rc = run([sys.executable, "-u", "-m", "auditorynet.train", "--config", CFG, "--tasks", t])
    if rc != 0:
        failed.append(t)
        print(f"!! Task {t} failed (exit code {rc})")
print("\nDone. Failed tasks:", failed or "none")

## 7) Combine results of all tasks

In [ ]:
import json
from auditorynet.plots import plot_cross_task

results = []
for t in TASKS:
    p = Path(OUTPUT_DIR) / f"{t}_results.json"
    if p.exists():
        results.append(json.loads(p.read_text()))
assert results, "No results found - training did not finish for any task."

rows = ["task,n_hc,n_pd,mean_acc,std_acc,mean_auc,std_auc,sensitivity,specificity,train_acc,train_test_gap"]
for r in results:
    g = r["aggregate"]
    rows.append(f"{r['task']},{r['n_hc']},{r['n_pd']},{r['mean_accuracy']:.4f},{r['std_accuracy']:.4f},"
                f"{r['mean_auc']:.4f},{r['std_auc']:.4f},{g['sensitivity']:.4f},{g['specificity']:.4f},"
                f"{r['mean_train_acc']:.4f},{r['train_test_gap']:.4f}")
(Path(OUTPUT_DIR) / "summary.csv").write_text("\n".join(rows) + "\n")
plot_cross_task(results, Path(OUTPUT_DIR) / "all_datasets_comparison.png")

summary = pd.read_csv(Path(OUTPUT_DIR) / "summary.csv")
display(summary)
for _, r in summary.iterrows():
    if r["train_test_gap"] > 0.10:
        print(f"WARNING task {r['task']}: train-test accuracy gap {r['train_test_gap']*100:.1f} points -> overfitting")

## 8) Look at the plots

In [ ]:
from IPython.display import Image, display
display(Image(str(Path(OUTPUT_DIR) / "all_datasets_comparison.png")))
for r in results:
    for name in (f"{r['task']}_5fold_results.png", f"{r['task']}_history.png"):
        f = Path(OUTPUT_DIR) / name
        if f.exists():
            print(name)
            display(Image(str(f)))

## 9) Download everything to your PC
Then unzip it into your project folder on your computer so you have `outputs/checkpoints/...`, and start `auditorynet-gui`.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/auditorynet_outputs", "zip", OUTPUT_DIR)
print("Also saved in Drive:", OUTPUT_DIR)
files.download(archive)

## 10) (Optional) Test a single recording

In [ ]:
from google.colab import files
TASK_FOR_PREDICTION = TASKS[0]      # must match what the person said (E, petaka, ...)
up = files.upload()                 # choose a .wav file
for name in up:
    run([sys.executable, "-m", "auditorynet.predict", name, "--task", TASK_FOR_PREDICTION,
         "--ckpt-dir", str(Path(OUTPUT_DIR) / "checkpoints")])